In [0]:
# COMMAND ----------
from pyspark.sql import functions as F

# Base S3 paths
S3_RAW_PATH = "s3://zubair-s3-demo/raw_dataset/aml"
S3_DELTA_PATH = "s3://zubair-s3-demo/raw_dataset/aml/delta_tables"
CHECKPOINT_PATH = f"{S3_RAW_PATH}/checkpoints"

# 1. Accounts Master Dimension (Saved to S3 External Location)
accounts_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{S3_RAW_PATH}/accounts/")
)

(
    accounts_df.write.format("delta")
    .mode("overwrite")
    .option("path", f"{S3_DELTA_PATH}/bronze_accounts")
    .saveAsTable("aml_engine.aml_poc.bronze_accounts")
)

# 2. Alerts Ground Truth (Saved to S3 External Location)
alerts_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{S3_RAW_PATH}/alerts/")
)

(
    alerts_df.write.format("delta")
    .mode("overwrite")
    .option("path", f"{S3_DELTA_PATH}/bronze_alerts")
    .saveAsTable("aml_engine.aml_poc.bronze_alerts")
)

# 3. Transactions Stream (Saved to S3 External Location)
raw_tx_stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT_PATH}/schema_tx")
    .load(f"{S3_RAW_PATH}/transactions/")
)

bronze_tx_df = (
    raw_tx_stream
    .withColumn("_ingested_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

query = (
    bronze_tx_df.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{CHECKPOINT_PATH}/bronze_tx")
    .option("path", f"{S3_DELTA_PATH}/bronze_transactions")
    .trigger(availableNow=True)
    .toTable("aml_engine.aml_poc.bronze_transactions")
)

query.awaitTermination()

In [0]:
%sql
SELECT 'bronze_transactions' AS table_name, count(*) AS total_records FROM aml_engine.aml_poc.bronze_transactions
UNION ALL
SELECT 'bronze_accounts', count(*) FROM aml_engine.aml_poc.bronze_accounts
UNION ALL
SELECT 'bronze_alerts', count(*) FROM aml_engine.aml_poc.bronze_alerts;